In [1]:
#Importing relevant libraries
import numpy as np
import pandas as pd
import random
from numpy.random import rand 
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lime.lime_tabular import LimeTabularExplainer
import shap


random_state = 42
import warnings

#suppress future warnings
warnings.filterwarnings("ignore", category = FutureWarning)

In [2]:
#Now we import out crop yeild dataset
url_data = 'Dataset/crop-yield.csv' #specifying the URL of the dataset

data = pd.read_csv(url_data)
data.head(5) #Displaying the first n rows from our dataset

,N(Nitrogen),P(Phosphorus),K(Potassium),Soil_pH,Soil_Moisture,Soil_Type,Organic_Carbon,Temperature,Humidity,Rainfall,Sunlight_Hours,Wind_Speed,Region,Altitude,Season,Crop_Type,Irrigation_Type,Fertilizer_Used,Pesticide_Used,Crop_Yield_ton_per_hectare
0,132,62,22,6.35,59.78,Clay,0.43,22.97,53.89,1305.68,7.73,15.96,Central,36,Rabi,Maize,Canal,223.48,23.36,11.42
1,122,71,66,5.98,25.54,Sandy,0.65,17.00,76.90,1942.05,9.25,12.60,North,1561,Rabi,Potato,Canal,161.54,4.42,23.19
2,44,35,104,8.07,25.87,Sandy,0.79,25.52,44.78,2216.20,8.50,15.63,North,1870,Rabi,Rice,Rainfed,184.62,6.29,7.94
3,136,96,113,4.83,42.97,Silt,0.45,18.59,31.89,607.18,8.75,5.49,East,765,Kharif,Sugarcane,Rainfed,274.02,2.72,72.53
4,101,34,42,5.84,48.01,Silt,0.69,22.74,46.27,483.47,8.00,7.44,Central,1143,Zaid,Wheat,Rainfed,72.69,15.37,6.72


In [3]:
data.shape 
'''checking the shape of your dataset. We can see that there are 10,000 instances 
and 20 features. Although it's 19 features with a target class''';

In [4]:
data.isnull().sum() #Checking for missing values. There are none!

N(Nitrogen)                   0
P(Phosphorus)                 0
K(Potassium)                  0
Soil_pH                       0
Soil_Moisture                 0
Soil_Type                     0
Organic_Carbon                0
Temperature                   0
Humidity                      0
Rainfall                      0
Sunlight_Hours                0
Wind_Speed                    0
Region                        0
Altitude                      0
Season                        0
Crop_Type                     0
Irrigation_Type               0
Fertilizer_Used               0
Pesticide_Used                0
Crop_Yield_ton_per_hectare    0
dtype: int64

In [5]:
data.dtypes #chceking the datatypes. 5 of them are categorical values

N(Nitrogen)                     int64
P(Phosphorus)                   int64
K(Potassium)                    int64
Soil_pH                       float64
Soil_Moisture                 float64
Soil_Type                      object
Organic_Carbon                float64
Temperature                   float64
Humidity                      float64
Rainfall                      float64
Sunlight_Hours                float64
Wind_Speed                    float64
Region                         object
Altitude                        int64
Season                         object
Crop_Type                      object
Irrigation_Type                object
Fertilizer_Used               float64
Pesticide_Used                float64
Crop_Yield_ton_per_hectare    float64
dtype: object

In [6]:
#Now we take out our target class away from the features
features = data.drop(['Crop_Yield_ton_per_hectare'], axis = 1)
target = data.loc[:,['Crop_Yield_ton_per_hectare']]

len(features.columns) #Checking number of features 

19

In [7]:
#Now we apply the LabelEncoder to convert the categorical values into numeric
#List of columns to encode
categorical_cols = ['Soil_Type', 'Region','Season','Crop_Type', 'Irrigation_Type']

Label_encoder = {}
for col in categorical_cols:
    le = LabelEncoder()
    features[col] = le.fit_transform(data[col])
    Label_encoder[col] = le
features.head(4)# checkking our converted features

,N(Nitrogen),P(Phosphorus),K(Potassium),Soil_pH,Soil_Moisture,Soil_Type,Organic_Carbon,Temperature,Humidity,Rainfall,Sunlight_Hours,Wind_Speed,Region,Altitude,Season,Crop_Type,Irrigation_Type,Fertilizer_Used,Pesticide_Used
0,132,62,22,6.35,59.78,0,0.43,22.97,53.89,1305.68,7.73,15.96,0,36,1,1,0,223.48,23.36
1,122,71,66,5.98,25.54,2,0.65,17.00,76.90,1942.05,9.25,12.60,2,1561,1,2,0,161.54,4.42
2,44,35,104,8.07,25.87,2,0.79,25.52,44.78,2216.20,8.50,15.63,2,1870,1,3,2,184.62,6.29
3,136,96,113,4.83,42.97,3,0.45,18.59,31.89,607.18,8.75,5.49,1,765,0,4,2,274.02,2.72


In [8]:
#Now we normalize/scale our dataset
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)
#visualizing our scaled features
features_scaled

array([[ 0.62410068,  0.21166527, -1.67846106, ..., -1.33180398,
         0.20924607,  1.50172122],
       [ 0.39309773,  0.57868659, -0.5063644 , ..., -1.33180398,
        -0.53162987, -1.12348403],
       [-1.40872532, -0.88939869,  0.50590089, ...,  0.46283822,
        -0.25556567, -0.86429006],
       ...,
       [-0.11510877,  1.7205307 , -1.62518394, ..., -0.43448288,
         1.27678202,  0.86828997],
       [-0.85431823, -1.50110089, -0.58628008, ...,  0.46283822,
         0.25733004, -0.2696686 ],
       [-0.16130936,  0.74180718, -1.19896697, ..., -1.33180398,
        -0.87395905, -0.71459515]])

In [9]:
#Just to see the scaled features in a tabular form
features_table = pd.DataFrame(features_scaled, columns = features.columns)
features_table.head(4)

,N(Nitrogen),P(Phosphorus),K(Potassium),Soil_pH,Soil_Moisture,Soil_Type,Organic_Carbon,Temperature,Humidity,Rainfall,Sunlight_Hours,Wind_Speed,Region,Altitude,Season,Crop_Type,Irrigation_Type,Fertilizer_Used,Pesticide_Used
0,0.624101,0.211665,-1.678461,-0.157673,1.362005,-1.347373,-1.408919,-0.244007,-0.345038,-0.335288,0.110713,1.314597,-1.419143,-1.676964,0.006747,-0.878728,-1.331804,0.209246,1.501721
1,0.393098,0.578687,-0.506364,-0.535745,-1.005707,0.447091,-0.822265,-0.928751,0.977256,0.549998,0.868300,0.630506,-0.002479,0.719019,0.006747,-0.292558,-1.331804,-0.531630,-1.123484
2,-1.408725,-0.889399,0.505901,1.599853,-0.982887,0.447091,-0.448940,0.048471,-0.868554,0.931382,0.494491,1.247410,-0.002479,1.204500,0.006747,0.293613,0.462838,-0.255566,-0.864290
3,0.716502,1.598190,0.745648,-1.710835,0.199586,1.344323,-1.355587,-0.746382,-1.609291,-1.307006,0.619094,-0.817080,-0.710811,-0.531605,-1.220021,0.879783,0.462838,0.813764,-1.359115


In [10]:
#convert the target variable into a 1-D array, which is what most scikit-learn regression models expect
target = np.ravel(target)
#target

In [11]:
#Now we spilt our dataset into train and test (80:20)
x_train, x_test, y_train, y_test = train_test_split(features_scaled,
                                                   target,
                                                   test_size = .2,
                                                   random_state = random_state)
fold = {'xt': x_train, 'yt': y_train, 'xv': x_test, 'yv':y_test}
#Chceking the distribution of dataset
print(f'Training set:{x_train.shape}')
print(f'Testing set:{x_test.shape}')
print(f'Training_Label set:{y_train.shape}')
print(f'Testing_Lable set:{y_test.shape}')

Training set:(8000, 19)
Testing set:(2000, 19)
Training_Label set:(8000,)
Testing_Lable set:(2000,)


#### Now we employ our Feature Selection using Firefly Algorithm (FA) with OBL (FA-OBL)

#### Pseudorandom-OBL Initialization

In [12]:
def init_position_Rand_OBL(lb, ub, N, dim, fitness_func):
    """
    Hybrid Pseudorandom-OBL initialization for population.
    """

    # Step 1: Generate pseudorandom-initialized population
    X = np.zeros((N, dim), dtype=float)
    for i in range(N):
        for d in range(dim):
            r = np.random.rand()  # Uniform pseudorandom number in [0,1]
            X[i, d] = lb[d] + (ub[d] - lb[d]) * r

    # Step 2: Generate oppositional population using OBL
    X_opposite = lb + ub - X

    # Step 3: Combine both populations
    X_combined = np.vstack((X, X_opposite))

    # Step 4: Evaluate fitness for all individuals
    fitness_values = np.array([fitness_func(ind) for ind in X_combined])

    # Step 5: Select top N individuals
    best_indices = fitness_values.argsort()  # minimization
    X_init = X_combined[best_indices[:N], :]

    return X_init

In [13]:
def fitness_func(x, opts):
    """
    Fitness wrapper for rand-OBL initialization
    Converts continuous vector to binary and evaluates using Fun()
    """
    thres = 0.6
    x_bin = binary_conversion(x.reshape(1, -1), thres)[0]

    try:
        return Fun(x_bin, opts)
    except ValueError:
        return 1e6  # heavy penalty if no feature is selected

In [14]:
def binary_conversion(X, thres):
    """
    Convert continuous values in X to binary using sigmoid and threshold.

    """
    Xbin = (1 / (1 + np.exp(-X)) > thres).astype(int)
    return Xbin

In [15]:
def boundary(x, lb, ub):
    """
making our search space to remian between 0 and 1
    """
    return np.clip(x, lb, ub)

In [16]:
def error_rate(x, opts):
    """
Error rate
    """
    k    = opts['k']
    fold = opts['fold']
    xt   = fold['xt']
    yt   = fold['yt'].ravel()
    xv   = fold['xv']
    yv   = fold['yv'].ravel()

    # Ensure at least one feature is selected
    if np.sum(x) == 0:
        raise ValueError("No features selected in x. At least one feature must be 1.")

    # Select features
    xtrain = xt[:, x == 1]
    xvalid = xv[:, x == 1]

    # Train KNN regressor
    mdl = KNeighborsRegressor(n_neighbors=k)
    mdl.fit(xtrain, yt)

    # Predict and compute MAE
    ypred = mdl.predict(xvalid)
    mae = mean_absolute_error(yv, ypred)

    return mae

In [17]:
def Fun(x, opts):
    """
Error rate and feature size
    """
    alpha = 0.8
    beta  = 1 - alpha

    max_feat = len(x)
    num_feat = np.sum(x == 1)

    if num_feat == 0:
        cost = 1  # Penalty if no features selected
    else:
        error = error_rate(x, opts)  # Compute MAE for selected features
        cost = alpha * error + beta * (num_feat / max_feat)

    return cost

In [18]:
def jfs(xtrain, ytrain, opts):
    """
main body of code
    """
    # Parameters
    N        = opts['N']
    max_iter = opts['T']
    alpha    = opts.get('alpha', 1)
    beta0    = opts.get('beta0', 1)
    gamma    = opts.get('gamma', 1)
    theta    = opts.get('theta', 0.97)
    thres    = 0.6

    dim = xtrain.shape[1]

    lb = np.zeros(dim)
    ub = np.ones(dim)

    # Initialize positions and binary representation
    X = init_position_Rand_OBL(lb, ub, N, dim, lambda x: fitness_func(x, opts))
    Xbin = binary_conversion(X, thres)

    # Evaluate initial fitness
    fit  = np.zeros(N)
    Xgb  = X[0,:].copy()
    fitG = float('inf')

    for i in range(N):
        fit[i] = Fun(Xbin[i,:], opts)
        if fit[i] < fitG:
            Xgb = X[i,:].copy()
            fitG = fit[i]

    # Store best fitness over iterations
    curve = np.zeros(max_iter)
    curve[0] = fitG
    print(f"Generation: 1, Best (FA): {fitG:.6f}")

    # Main loop
    for t in range(1, max_iter):
        alpha = alpha * theta

        # Rank fireflies by fitness
        ind = np.argsort(fit)
        X   = X[ind,:]
        fit = fit[ind]

        for i in range(N):
            for j in range(N):
                if fit[i] > fit[j]:
                    # Distance
                    r = np.linalg.norm(X[i,:] - X[j,:])
                    # Attractiveness
                    beta = beta0 * np.exp(-gamma * r**2)
                    # Move firefly
                    eps = np.random.rand(dim) - 0.5
                    X[i,:] = X[i,:] + beta * (X[j,:] - X[i,:]) + alpha * eps
                    # Boundary
                    X[i,:] = boundary(X[i,:], lb, ub)
                    # Binary conversion
                    xbin_i = binary_conversion(X[i,:].reshape(1,-1), thres)[0]
                    # Fitness
                    fit[i] = Fun(xbin_i, opts)
                    # Update global best
                    if fit[i] < fitG:
                        Xgb = X[i,:].copy()
                        fitG = fit[i]

        curve[t] = fitG
        print(f"Generation: {t+1}, Best (FA): {fitG:.6f}")

    # Final best solution
    Gbin      = binary_conversion(Xgb.reshape(1,-1), thres)[0]
    sel_index = np.where(Gbin == 1)[0]
    num_feat  = len(sel_index)

    return {'sf': sel_index, 'c': curve, 'nf': num_feat}

In [19]:
#Now we set the general parameters
k = 5 #K value in KNN
N = 20 #Population size
T = 50 #Max iteration/generations
alpha  = 1       # constant
beta0  = 1       # light amplitude
gamma  = 1       # absorbtion coefficient
theta  = 0.97    # control alpha

opts = {'k':k, 'fold':fold, 'N':N, 'T':T, 'alpha':alpha, 'beta0':beta0, 'gamma':gamma, 'theta':theta}

In [ ]:
#Now we perform our feature selection by calling the module jfs
feature_selection_mdl = jfs(features_scaled, target, opts) #feature selection model
sf = feature_selection_mdl['sf'] #index of selected features

In [ ]:
print('The number of features Selected:', len(sf)) #Checking how many features got selected 

In [ ]:
#Model with selected features
num_train = np.size(x_train,0)#checks the number of samples slected in the x_train and assign to num_train
num_test = np.size(x_test, 0)#checks the number of samples slected in the x_test and assign to num_test
xtrain = x_train[:,sf]
ytrain = y_train.reshape(num_train)
xtest = x_test[:,sf]
ytest = y_test.reshape(num_test)

In [ ]:
# Build KNN regression model
knn_model = KNeighborsRegressor(n_neighbors=5)  # you can adjust k
knn_model.fit(xtrain, ytrain)  # train the model

# Make predictions on the test set
ypred = knn_model.predict(xtest)

# Evaluation
mae = mean_absolute_error(ytest, ypred)
rmse = np.sqrt(mean_squared_error(ytest, ypred))

print(f"Mean Absolute Error (MAE): {mae:.4f}")
print('')
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")

#### Now we use our SHAP Explanable AI to understand our predictions (Run this code only on one instance of your run).

In [ ]:
# Explain predictions using Kernel SHAP (model-agnostic)

# Your features (exclude target column)
feature_columns = [col for col in data.columns if col != 'Crop_Yield_ton_per_hectare']

# Scaled features for model
features_scaled = data[feature_columns].values

# Selected feature names (after FA)
selected_feature_names = [feature_columns[i] for i in sf]

# Small background sample (only selected features)
background = shap.sample(xtrain, 500) 

# KernelExplainer
explainer = shap.KernelExplainer(knn_model.predict, background)

# Compute SHAP values for test set
shap_values = explainer.shap_values(xtest)

# Plot summary
shap.summary_plot(shap_values, xtest, feature_names=selected_feature_names)

### END OF PROJECT